# Survival analysis: time-to-event models

Exercise notebook for the lesson **[Survival analysis: time-to-event models](https://ramkumarmd.com/medical-ai/machine-learning/survival-analysis)** in the *Machine learning* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Survival analysis models the time until an event, such as kidney failure, while correctly handling patients whose follow-up ended before any event (censoring). Simply counting events among everyone underestimates risk because some patients have not been followed long enough. Kaplan-Meier curves estimate risk over time, Cox models relate risk to patient characteristics, and both can give a predicted risk at a chosen time, such as five years.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q lifelines

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A synthetic kidney-failure cohort

Synthetic time to kidney failure with censoring at varying follow-up times.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(21)
n = 3000
df = pd.DataFrame({
    "age": rng.integers(40, 80, n),
    "egfr": rng.normal(60, 18, n).clip(15, 120).round(),   # mL/min/1.73m²
    "uacr_high": rng.binomial(1, 0.35, n),                  # albuminuria
    "sglt2i": rng.binomial(1, 0.4, n),
})
hazard = 0.02 * np.exp(0.02 * (df.age - 60) - 0.04 * (df.egfr - 60)
                       + 0.8 * df.uacr_high - 0.3 * df.sglt2i)
event_time = rng.exponential(1 / hazard)
censor_time = rng.uniform(1, 8, n)           # end of follow-up varies by patient
df["years"] = np.minimum(event_time, censor_time).round(2)
df["kidney_failure"] = (event_time <= censor_time).astype(int)
print(f"{df.kidney_failure.sum()} events among {n} patients; "
      f"{(1 - df.kidney_failure.mean()):.0%} censored")

# 368 events among 3000 patients; 88% censored

### Kaplan-Meier: risk over time

Continuing from the previous block. Install lifelines first if needed: pip install lifelines

In [ ]:
from lifelines import KaplanMeierFitter

naive = df[df.years <= 5].kidney_failure.sum() / n
print(f"Naive 'events / everyone' by 5 years: {naive:.1%}")

kmf = KaplanMeierFitter().fit(df.years, df.kidney_failure)
print(f"Kaplan-Meier risk by 5 years:        {1 - kmf.predict(5):.1%}")

# Naive 'events / everyone' by 5 years: 10.9%
# Kaplan-Meier risk by 5 years:        13.8%

### Cox proportional-hazards regression

Continuing from the previous block: fit a Cox model and show hazard ratios with 95% confidence intervals.

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter().fit(df, duration_col="years", event_col="kidney_failure")
print(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"]].round(2))
print(f"\nConcordance index {cph.concordance_index_:.3f}")

#            exp(coef)  exp(coef) lower 95%  exp(coef) upper 95%
# covariate
# age             1.02                 1.01                 1.02
# egfr            0.96                 0.95                 0.96
# uacr_high       2.77                 2.25                 3.41
# sglt2i          0.72                 0.58                 0.89
#
# Concordance index 0.737

### From hazard ratios to individual risk

Continuing from the previous block: predicted 5-year risk for two synthetic patients.

In [ ]:
patients = pd.DataFrame({"age": [55, 70], "egfr": [80, 35],
                         "uacr_high": [0, 1], "sglt2i": [0, 0]})
risk_5y = 1 - cph.predict_survival_function(patients, times=[5]).iloc[0]
print(risk_5y.round(3).to_string())

# 0    0.034
# 1    0.559

## Exercise

**Goal:** Analyse a synthetic kidney-failure cohort with survival methods, and see how censoring and competing risks change the answer.

**Data:** Synthetic cohort generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the naive and Kaplan-Meier five-year risks, and plot Kaplan-Meier curves by albuminuria status.

In [ ]:
# Your code here


### Step 2

Fit the Cox model, check the proportional-hazards assumption, and report hazard ratios with confidence intervals.

In [ ]:
# Your code here


### Step 3

Add a competing risk of death to the simulation and compare 1 − Kaplan-Meier with the Aalen-Johansen cumulative incidence of kidney failure.

In [ ]:
# Your code here


### Step 4

Calculate the C-index on a held-out test set and draw a calibration plot of predicted versus observed five-year risk.

In [ ]:
# Your code here


### Step 5

Fit a random survival forest (scikit-survival) and compare its C-index with the Cox model.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/survival-analysis](https://ramkumarmd.com/medical-ai/machine-learning/survival-analysis). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.